# SV model with normal-mixture approximation

本 Notebook 演示如何在本地数据文件夹中加载分钟级期货数据，使用混合正态近似的随机波动（SV）模型进行快速 MCMC 演示。所有注释使用中文，print 输出和图表元素使用英文，便于跨平台显示。

In [ ]:
# 导入依赖，注释使用中文
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# 可视化与进度显示
import matplotlib.pyplot as plt
import seaborn as sns

# 自定义模块
from sv_toolkit.data import (
    list_csv_files,
    load_contracts_in_dir,
    get_contract_symbol_from_path,
    load_single_file,  # 新增：单文件加载函数
)
from sv_toolkit.mcmc import run_mcmc_sv
from sv_toolkit.batch import make_timestamped_root, save_param_summary
from sv_toolkit.plotting import (
    plot_returns,
    plot_volatility,
    plot_return_histogram,
    plot_acf_returns,
    plot_intraday_pattern,
    plot_param_posterior,
    plot_vol_and_abs_returns,
    plot_standardized_residuals,
    plot_mixture_usage,
)

# 设置 matplotlib 显示风格
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# 确定数据目录，默认为当前仓库下的 2005年__20250905 文件夹
data_dir = Path('../2005年__20250905')

# 创建带时间戳的输出目录，方便保存图片与参数
output_root = make_timestamped_root(Path('outputs'))
fig_dir = output_root / 'demo'
fig_dir.mkdir(parents=True, exist_ok=True)

print(f'Environment ready. Data dir: {data_dir}. Output root: {output_root}')


## 1. 浏览数据文件



In [ ]:
# 列出数据目录中的前 5 个文件，避免一次性打印全部
preview_files = list_csv_files(data_dir, max_files=5)
print('Preview finished.')

## 2. 读取单个/少量文件并配置路径


In [ ]:
# ===== Step 1: 配置要批量处理的合约文件名 =====

# 想要一次性处理的 5 个文件名
target_files = [
    "AG_主力合约_1m数据.csv",
    "AL_主力合约_1m数据.csv",
    "AO_主力合约_1m数据.csv",
    "AP_主力合约_1m数据.csv",
    "AU_主力合约_1m数据.csv",
]

# 列出目录下所有 csv，并建立 name -> Path 的映射
all_csv = list_csv_files(data_dir, max_files=200)
name_to_path = {p.name: p for p in all_csv}

selected_paths = []
for name in target_files:
    if name not in name_to_path:
        print(f"Warning: {name} not found under {data_dir}")
    else:
        selected_paths.append(name_to_path[name])

if not selected_paths:
    raise RuntimeError("None of the target csv files were found in the data directory.")

print("Selected paths:")
for p in selected_paths:
    print(" -", p)


## 3. 创建目录并初始化数据集


In [ ]:
# ===== Step 2: 创建批量输出目录 & 初始化容器 =====

# 批量结果放在 output_root 下的一个子目录里
batch_out_dir = output_root / "batch_multi_contracts"
batch_out_dir.mkdir(parents=True, exist_ok=True)
print("Batch output dir:", batch_out_dir)

# 这个 datasets 会被后面代码使用（存每个合约的 r / y_star / df / exog）
datasets = {}


## 4. 对每个合约跑mcmc，画图，存参数


In [ ]:
# ===== Step 3: 对每个合约依次运行 SV MCMC 并输出结果 =====

for file_path in selected_paths:
    symbol = get_contract_symbol_from_path(file_path)
    print(f"
=== Running batch SV for file {file_path.name} (symbol = {symbol}) ===")

    contract_out_dir = batch_out_dir / symbol
    contract_out_dir.mkdir(parents=True, exist_ok=True)

    # 加载该合约的数据（这里不截时间、不限制行数，你可以按需改 max_rows 等）
    r, y_star, df, exog = load_single_file(
        file_path,
        contract_code=None,
        start_time=None,
        end_time=None,
        max_rows=5000,          # 如需更长区间可以改为 None 或更大
        sample_every=1,
        state_exog_col=None,
        log_exog=True,
    )

    # 运行一遍轻量级 MCMC
    mcmc_results = run_mcmc_sv(
        r=r,
        y_star=y_star,
        n_iter=120,
        burn_in=40,
        thin=2,
        rng_seed=2025,
        progress_every=20,
        # 若以后要加 exog，可以传 exog_state=exog
    )

    # 保存参数摘要
    extra_info = {
        "file_name": file_path.name,
        "contract_tag": symbol,
        "T": len(r),
        "n_iter": 120,
        "burn_in": 40,
        "thin": 2,
        "data_dir": str(data_dir),
    }
    save_param_summary(mcmc_results, contract_out_dir, symbol, extra_info=extra_info)

    # 画图：输出到每个合约自己的文件夹
    returns_png = plot_returns(df, contract_out_dir, title_suffix=symbol)
    print(f"[{symbol}] Returns figure saved to: {returns_png}")

    if len(mcmc_results["h"]) > 0:
        vol_png = plot_volatility(mcmc_results["h"], df, contract_out_dir, title_suffix=symbol)
        print(f"[{symbol}] Volatility figure saved to: {vol_png}")
    else:
        print(f"[{symbol}] No h samples available to plot volatility.")

    hist_png = plot_return_histogram(r, contract_out_dir, title_suffix=symbol)
    acf_png = plot_acf_returns(r, contract_out_dir, title_suffix=symbol)
    intra_png = plot_intraday_pattern(df, contract_out_dir, title_suffix=symbol)
    param_png = plot_param_posterior(mcmc_results, contract_out_dir, title_suffix=symbol)
    vol_abs_png = plot_vol_and_abs_returns(mcmc_results["h"], df, contract_out_dir, title_suffix=symbol)
    std_resid_png = plot_standardized_residuals(r, mcmc_results, contract_out_dir, title_suffix=symbol)
    mix_png = None
    if "s" in mcmc_results and len(mcmc_results["s"]) > 0:
        mix_png = plot_mixture_usage(mcmc_results["s"], contract_out_dir, title_suffix=symbol)

    print(
        f"[{symbol}] Additional figures:",
        hist_png,
        acf_png,
        intra_png,
        param_png,
        vol_abs_png,
        std_resid_png,
        mix_png,
    )

    # 把当前合约的数据放入 datasets，给后续使用
    datasets[symbol] = {
        "r": r,
        "y_star": y_star,
        "df": df,
        "exog": exog,
    }

print(f"
Finished batch processing for {len(datasets)} contracts: {list(datasets.keys())}")


## 5. 绘制收益率、隐含波动率与其他诊断图



In [ ]:
# 上述循环已完成所有合约的输出。可以在此处添加自定义汇总。
